## Dilan Shah - Cycle 4 API Notebook
The data in this notebook comes from PokeAPI which includes data about everything Pokemon related. The API does not use a key and does not require any authentication. I used the following 2 endpoints to explore:

---

**Endpoint 1:** https://pokeapi.co/api/v2/location/canalave-city/ 

**Route parameter:** `canalave-city` (can be replaced with any known Pokemon location or ID)

**Description:** This retrieves information about the Pokemon location Canalave City such as the ID, names, region, areas found in the location, and games related to the location.

---

**Endpoint 2:** https://pokeapi.co/api/v2/pokemon-color/green/

**Route parameter:** `green` (can be replaced with any color name)

**Description:** This retrieves information about all the species related to the color, its id, and the color in various languages. 


In [1]:
import json

with open('./api-fixtures/location-fixture.json', encoding="utf-8") as handle:
    location=json.load(handle)

with open('./api-fixtures/color-fixture.json', encoding="utf-8") as handle:
    color=json.load(handle)

print(location.keys())
print(color.keys())

dict_keys(['id', 'name', 'region', 'names', 'game_indices', 'areas'])
dict_keys(['id', 'name', 'names', 'pokemon_species'])


Above, I open the fixtures that I downloaded as json files from the PokeAPI website. I store one fixture in the variable `location`, and the other as `color`. Then, I printed the dictionaries' keys. 

Keys for Location endpoint: id, name, region, names, game_indices, areas

Keys for Color endpoint: id, name, names, pokemon_species

## API Connection

In [2]:
ENDPOINT1 = "https://pokeapi.co/api/v2/location/canalave-city/"
ENDPOINT2 = 'https://pokeapi.co/api/v2/pokemon-color/green/'

The two functions below ensure that the data from PokeAPI loads as expected. They check for correct keys, datatypes, and expected lengths or values to match the fixture stored in the json with live data.

In [3]:
def validate_location(payload):
    if not isinstance(payload, dict): #seeing if the payload is a dictionary (type checker)
        raise TypeError("API payload must be a JSON object") 

    if "region" not in payload or not isinstance(payload["region"], dict): #checking if the key I want exists
        raise ValueError("Missing region dict")

    #comparing expected keys with actual keys 
    expected={'name','url'}
    missing=expected-set(payload['region'])
    if missing:
        raise ValueError(f"Missing daily fields: {sorted(missing)}") # checking if any fields in keys list aren't in the live

    if "names" not in payload or not isinstance(payload["names"], list): #checking if the key I want exists
        raise ValueError("Missing names list")

    # checking that list has expected length
    if len(payload['names']) != 3:
        raise ValueError("Not expected length of list")

    return payload

In [4]:
def validate_color(payload):
    if not isinstance(payload, dict): #seeing if the payload is a dictionary (type checker)
        raise TypeError("API payload must be a JSON object") 

    if "pokemon_species" not in payload or not isinstance(payload["pokemon_species"], list): #checking if the key I want exists
        raise ValueError("Missing species list")

    # checking that list has expected length
    if len(payload['pokemon_species']) != 130:
        raise ValueError("Not expected length of list")

    return payload

This function below loads the live data from the PokeAPI and can be used for both endpoints to call for the data. It takes in 4 parameters: the fixture, endpoint, validation function, and a boolean that checks if you want to use live data. By default, it uses the fixture data. When `use_live` is set true, it checks the HTTP status, requests the live endpoint, and validates before returning the live payload.

In [5]:
def acquire_pokemon(fixture,endpoint,func,use_live=False):
    source = "cached fixture"
    payload = fixture #setting the payload equal to the fixutre, and is use_live is false, it uses the fixture
    if use_live:
        try: #stops code execution if an exception is raised in validate_weather_payload function
            import requests
            response = requests.get(endpoint, timeout=10) #sending a get request using the requests library
            response.raise_for_status() #checks the http status code (200 means OK)
            payload = response.json() #converts response to JSON
            source = response.url # URl of the source
            func(payload) #validate code
        except (requests.RequestException, ValueError, TypeError) as exc:
            print(f"Live request unavailable ({type(exc).__name__}); using fixture.")
            payload = fixture
    return payload, source

In [6]:
location_payload, location_source = acquire_pokemon(location,ENDPOINT1,validate_location,use_live=True)
location_payload

{'id': 1,
 'name': 'canalave-city',
 'region': {'name': 'sinnoh', 'url': 'https://pokeapi.co/api/v2/region/4/'},
 'names': [{'name': 'Joliberges',
   'language': {'name': 'fr', 'url': 'https://pokeapi.co/api/v2/language/5/'}},
  {'name': 'Fleetburg',
   'language': {'name': 'de', 'url': 'https://pokeapi.co/api/v2/language/6/'}},
  {'name': 'Canalave City',
   'language': {'name': 'en',
    'url': 'https://pokeapi.co/api/v2/language/9/'}}],
 'game_indices': [{'game_index': 7,
   'generation': {'name': 'generation-iv',
    'url': 'https://pokeapi.co/api/v2/generation/4/'}}],
 'areas': [{'name': 'canalave-city-area',
   'url': 'https://pokeapi.co/api/v2/location-area/1/'}]}

In [7]:
color_payload, color_source = acquire_pokemon(color,ENDPOINT2,validate_color,use_live=True)
color_payload

{'id': 5,
 'name': 'green',
 'names': [{'name': 'みどりいろ',
   'language': {'name': 'ja-hrkt',
    'url': 'https://pokeapi.co/api/v2/language/1/'}},
  {'name': '초록',
   'language': {'name': 'ko', 'url': 'https://pokeapi.co/api/v2/language/3/'}},
  {'name': '綠色',
   'language': {'name': 'zh-hant',
    'url': 'https://pokeapi.co/api/v2/language/4/'}},
  {'name': 'Vert',
   'language': {'name': 'fr', 'url': 'https://pokeapi.co/api/v2/language/5/'}},
  {'name': 'Grün',
   'language': {'name': 'de', 'url': 'https://pokeapi.co/api/v2/language/6/'}},
  {'name': 'Verde',
   'language': {'name': 'es', 'url': 'https://pokeapi.co/api/v2/language/7/'}},
  {'name': 'Verde',
   'language': {'name': 'it', 'url': 'https://pokeapi.co/api/v2/language/8/'}},
  {'name': 'Green',
   'language': {'name': 'en', 'url': 'https://pokeapi.co/api/v2/language/9/'}},
  {'name': '緑色',
   'language': {'name': 'ja',
    'url': 'https://pokeapi.co/api/v2/language/11/'}},
  {'name': '绿色',
   'language': {'name': 'zh-hans',

## Normalizing API Data as DataFrames
Below, I import the data from the PokeAPI and turn it into a pandas dataframe on the specific keys I wanted to use.
I used pd.json_normalize() to convert the json data in the names list into a DataFrame, turning the nested dictionaries into separate columns. Source: https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html

In [8]:
import pandas as pd
names = pd.json_normalize(location_payload['names'])
names

,name,language.name,language.url
0,Joliberges,fr,https://pokeapi.co/api/v2/language/5/
1,Fleetburg,de,https://pokeapi.co/api/v2/language/6/
2,Canalave City,en,https://pokeapi.co/api/v2/language/9/


In [9]:
species = pd.json_normalize(color_payload['pokemon_species'])
species

,name,url
0,bulbasaur,https://pokeapi.co/api/v2/pokemon-species/1/
1,ivysaur,https://pokeapi.co/api/v2/pokemon-species/2/
2,venusaur,https://pokeapi.co/api/v2/pokemon-species/3/
3,caterpie,https://pokeapi.co/api/v2/pokemon-species/10/
4,metapod,https://pokeapi.co/api/v2/pokemon-species/11/
...,...,...
125,dipplin,https://pokeapi.co/api/v2/pokemon-species/1011/
126,poltchageist,https://pokeapi.co/api/v2/pokemon-species/1012/
127,sinistcha,https://pokeapi.co/api/v2/pokemon-species/1013/
128,ogerpon,https://pokeapi.co/api/v2/pokemon-species/1017/


Now, I copied a previously used function to summarize the dataframe's qualities.

In [10]:
# function that summarizes data
def profile_table(frame):
    return {
        "shape": frame.shape,
        "columns": frame.columns.tolist(),
        "missing": frame.isna().sum().to_dict(),
        "types": frame.dtypes,
        "exact_duplicates": int(frame.duplicated().sum()),
    }

In [11]:
profile_table(names)

{'shape': (3, 3),
 'columns': ['name', 'language.name', 'language.url'],
 'missing': {'name': 0, 'language.name': 0, 'language.url': 0},
 'types': name             str
 language.name    str
 language.url     str
 dtype: object,
 'exact_duplicates': 0}

In [12]:
profile_table(species)

{'shape': (130, 2),
 'columns': ['name', 'url'],
 'missing': {'name': 0, 'url': 0},
 'types': name    str
 url     str
 dtype: object,
 'exact_duplicates': 0}

As expected, the length of the dataframes are equivalent to the length of the lists in the fixtures, and the keys match the column names. There are no missing values or duplicates. While this data was imported correctly, I think its use is limited as both tables provide a small amount of data. In a further analysis, the `species` data would be helpful to determine which types of pokemon species are green and what qualities they share. It would most likely need to be merged with another dataset including the Pokemon and their stats. 

## Provenance

In [13]:
provenance = {
    "provider": "Open-Meteo",
    "endpoint": [ENDPOINT1,ENDPOINT2],
    "acquisition_source": {'canalave city': location_source, 'green' : color_source},
    "fixture_retrieved": "2026-11-15",
    "Source": "PokeAPI",
}
provenance

{'provider': 'Open-Meteo',
 'endpoint': ['https://pokeapi.co/api/v2/location/canalave-city/',
  'https://pokeapi.co/api/v2/pokemon-color/green/'],
 'acquisition_source': {'canalave city': 'https://pokeapi.co/api/v2/location/canalave-city/',
  'green': 'https://pokeapi.co/api/v2/pokemon-color/green/'},
 'fixture_retrieved': '2026-11-15',
 'Source': 'PokeAPI'}